# Step-by-step demo of NEDAS using the vort2d model

## Load NEDAS and dependencies

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from NEDAS import get_scheme
from NEDAS.config import Config

## Initialize configuration and main scheme

In [2]:
config = Config(config_file="vort2d/config.yml")

In [3]:
scheme = get_scheme(config)

model = scheme.c.models['vort2d']
dataset = scheme.c.datasets['vort2d']

In [4]:
grid = scheme.c.grid

## Prepare truth

check nc files in vort2d/truth

In [5]:
scheme.c.time = config.time_start

In [11]:
%rm -rf vort2d/work/truth

In [13]:
model.loc_sprd = 0
scheme.run_step('prepare_truth')


Running prepare_truth step ────────────────────── ✅     0.03s (all truth data ready)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           


## Prepare initial ensemble and run forecasts

In [14]:
%rm -rf vort2d/work/init_ens

In [16]:
model.loc_sprd = 100000
scheme.run_step('prepare_init_ensemble')


Running prepare_init_ensemble step ────────────── ✅     0.03s (all initial ensemble states ready)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              


In [17]:
%rm -rf vort2d/work/cycle

In [18]:
scheme.c.progress.call_stack = []
scheme.c.progress.push('')
scheme.c.time = config.time_start
scheme.c.log_event("CYCLING START...")
while scheme.c.time < config.time_end:
    scheme.c.log_event(f"CURRENT CYCLE: {scheme.c.time} -> {scheme.c.next_time}")
    scheme.run_step('preprocess')
    scheme.run_step('ensemble_forecast')
    scheme.c.time = scheme.c.next_time
scheme.c.log_event("CYCLING COMPLETE.", flag='finish')


│   
◈ CYCLING START...
│   
◈ CURRENT CYCLE: 2001-01-01 00:00:00+00:00 -> 2001-01-01 03:00:00+00:00
├── Running preprocess step ───────────────────── ✅     0.60s (all 16 jobs done)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          
├── Running ensemble_forecast step ────────────── ✅     6.41s (all 16 jobs done)       

In [10]:
scheme.c.time = config.time_start
truth_state = []
times = []
while scheme.c.time < config.time_end:
    times.append(scheme.c.time)

    truth = model.read_var(path = model.truth_dir, name='velocity', member=None, time=scheme.c.time)
    truth_state.append(truth)
    scheme.c.time = scheme.c.next_time

In [11]:
scheme.c.time = config.time_start
forecast_state = []
for n in range(10):
    path = scheme.c.fs.forecast_dir(scheme.c.time, 'vort2d')
    ens = []
    for m in range(scheme.c.nens):
        ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
    forecast_state.append(ens)
    scheme.c.time = scheme.c.next_time

In [13]:
from NEDAS.utils.spatial_operation import gradx, grady

def to_vorticity(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
nt = 10 #len(truth_state)
clvs = [1e-3]
current_contour_sets = []

def update(n):
    global current_contour_sets
    for cs in current_contour_sets:
        cs.remove() 
    current_contour_sets.clear()

    pc_truth = ax[0].pcolor(to_vorticity(truth_state[n]), vmin=-5e-3, vmax=5e-3, cmap='bwr')
    current_contour_sets.append(pc_truth)
    cmap = [plt.cm.jet(x) for x in np.linspace(0, 1, scheme.c.nens)]
    for m in range(scheme.c.nens):
        cs_ens = ax[0].contour(to_vorticity(forecast_state[n][m]), clvs, colors=[cmap[m][0:3]], linewidths=0.5)
        current_contour_sets.append(cs_ens)
    cs_truth = ax[0].contour(to_vorticity(truth_state[n]), clvs, colors='k', linewidths=1.5)
    current_contour_sets.append(cs_truth)
    ax[0].set_title(f"Forecast {n*config.cycle_period} h")
    return current_contour_sets

ani = FuncAnimation(fig, update, frames=range(nt), blit=False, interval=100)
plt.close()
HTML(ani.to_jshtml())

## Substeps in an analysis cycle

In [12]:
scheme.c.time = config.time_start

## Running the entire scheme

Cycling from time_start to time_end

In [14]:
scheme.c.time = config.time_start
scheme.c.progress.call_stack = []

scheme()



█▄  █ █▀▀▀ █▀▀▄ ▄▀▀▄ ▄▀▀▀
█ ▀▄█ █▀▀  █  █ █▀▀█ ▀▀▀█
▀   ▀ ▀▀▀▀ ▀▀▀  ▀  ▀ ▀▀▀  version 1.2.1.dev2+ge38624e37

INITIALIZING...

CONFIGURATION SUMMARY
Directories:
  Work Dir:      /home/onyxia/work/NEDAS_tutorials/vort2d/work
  NEDAS Root:    /opt/python/lib/python3.13/site-packages/NEDAS

Time Configuration:
  Current Time:  2001-01-01 00:00:00+00:00
  Experiment:    [2001-01-01 00:00:00+00:00] to [2001-01-03 00:00:00+00:00]
  Analysis:      [2001-01-01 03:00:00+00:00] to [2001-01-02 21:00:00+00:00]
  Periods:       Cycle: 3h | Forecast: N/A

Parallel Scheme:
  Total Procs:   1
  Decomposition: 1 (mem) x 1 (rec)
  Procs for utility funcs: 8
  Host:          local
  Scheduler:     None | Project: N/A
  Queue/Mode:    N/A | Parallel mode: mpi

Analysis Scheme:
  General:       Scheme: filter | Ensemble Size: 16 | IO: offline
  Grid Type:     vort2d
  Iteration:     1 of 1 (Outer Loops)
  Assimilator:   Type: ETKF
  Updator:       Type: additive
  Inflation:     post,multiplicative (coef:

In [15]:
scheme.c.time = config.time_start
prior_state = []
post_state = []
while scheme.c.time < config.time_end:

    path = scheme.c.fs.forecast_dir(scheme.c.prev_time, 'vort2d')
    ens = []
    for m in range(scheme.c.nens):
        ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
    prior_state.append(ens)

    ens = []
    for m in range(scheme.c.nens):
        if config.run_analysis and scheme.c.time >= config.time_analysis_start and scheme.c.time <= config.time_analysis_end:
            path = scheme.c.fs.forecast_dir(scheme.c.time, 'vort2d')
            ens.append(model.read_var(path=path, name='velocity', member=m, time=scheme.c.time))
        else:
            ens.append(np.full(truth.shape, np.nan))
    post_state.append(ens)

    scheme.c.time = scheme.c.next_time

In [40]:
from NEDAS.utils.spatial_operation import gradx, grady

def to_vorticity(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
nt = 10 #len(truth_state)
clvs = [1e-3]
current_contour_sets = []

def update(n):
    global current_contour_sets
    for cs in current_contour_sets:
        cs.remove() 
    current_contour_sets.clear()

    pc_truth = ax[0].pcolormesh(to_vorticity(truth_state[n]), vmin=-5e-3, vmax=5e-3, cmap='bwr')
    current_contour_sets.append(pc_truth)
    cmap = [plt.cm.jet(x) for x in np.linspace(0, 1, scheme.c.nens)]
    for m in range(scheme.c.nens):
        cs_ens = ax[0].contour(to_vorticity(prior_state[n][m]), clvs, colors=[cmap[m][0:3]], linewidths=0.5)
        current_contour_sets.append(cs_ens)
    cs_truth = ax[0].contour(to_vorticity(truth_state[n]), clvs, colors='k', linewidths=1.5)
    current_contour_sets.append(cs_truth)
    ax[0].set_title(f"Forecast {n*config.cycle_period} h")
    return current_contour_sets

ani = FuncAnimation(fig, update, frames=range(nt), blit=False, interval=100)
plt.close()
HTML(ani.to_jshtml())

## A few things to try to play with
